# 크롤링

In [1]:
import requests

REST_API_KEY = "8cb01ceb3d70c35bae25270a6353bdc5"  # 카카오 API 인증 키임
KEYWORD_LOCAL_URL = "https://dapi.kakao.com/v2/local/search/keyword.json?query={}&radius=1000"  # 키워드로 맛집을 검색하는 URL 템플릿임
COMMENT_URL_TEMPLATE = 'https://place.map.kakao.com/m/commentlist/v/{}/{}?order=USEFUL&onlyPhotoComment=false'  # 리뷰를 가져오는 URL 템플릿임

headers = {
    "Authorization": f"KakaoAK {REST_API_KEY}"  # 인증 키를 헤더에 추가함
}

In [2]:
def get_restaurants(station_name):
    response = requests.get(KEYWORD_LOCAL_URL.format(station_name + " 맛집"), headers=headers)  # 맛집 검색 요청을 보냄

    if response.status_code == 200:  # 요청이 성공하면
        return response.json()['documents']  # 맛집 목록을 반환함
    else:
        return []  # 요청이 실패하면 빈 리스트를 반환함

In [3]:
# get_restaurants("강남역")

In [4]:
def get_reviews(place_id):
    all_comments = []  # 모든 리뷰를 저장할 리스트임

    comment_id = 0  # 첫 번째 코멘트의 ID는 0임

    has_next = True  # 다음 페이지가 있는지 여부를 나타내는 변수임

    while has_next:

        SCRAP_COMMENT_URL = COMMENT_URL_TEMPLATE.format(place_id, comment_id)  # 리뷰를 가져올 URL을 생성함

        response = requests.get(SCRAP_COMMENT_URL)  # 리뷰 데이터 요청을 보냄

        # if response.status_code != 200:  # 요청이 실패하면
        #     break  # 반복문을 종료함

        comment_datas = response.json()['comment']  # 리뷰 데이터를 가져옴

        comment_list = comment_datas['list']  # 리뷰 리스트를 추출함

        all_comments.extend(comment_list)  # 모든 리뷰 리스트에 추가함

        has_next = comment_datas['hasNext']  # 다음 페이지가 있는지 여부를 확인함

        if has_next:
            comment_id = comment_list[-1]['commentid']  # 다음 페이지가 있으면 마지막 리뷰의 ID를 설정함

    return all_comments  # 모든 리뷰를 반환함

In [5]:
# get_reviews("1238400864")

In [6]:
def collect_reviews(stations):
    all_reviews = []  # 모든 리뷰를 저장할 리스트임

    for station in stations:  # 각 지하철역에 대해
        response = requests.get(KEYWORD_LOCAL_URL.format(station + " 맛집"), headers=headers)  # 맛집 검색 요청을 보냄

        restaurants = response.json().get('documents', [])  # 맛집 목록을 가져옴
        print(restaurants)

        for restaurant in restaurants:  # 각 맛집에 대해
            place_id = restaurant['id']  # 맛집의 ID를 가져옴
            all_comments = []  # 모든 리뷰를 저장할 리스트임
            comment_id = 0  # 첫 번째 코멘트의 ID는 0임
            has_next = True  # 다음 페이지가 있는지 여부를 나타내는 변수임

            while has_next:
                SCRAP_COMMENT_URL = COMMENT_URL_TEMPLATE.format(place_id, comment_id)  # 리뷰를 가져올 URL을 생성함
                response = requests.get(SCRAP_COMMENT_URL)  # 리뷰 데이터 요청을 보냄
                print(response.json())

                if not response.json():
                    break

                comment_datas = response.json()['comment']  # 리뷰 데이터를 가져옴

                comment_list = comment_datas.get('list', [])  # 리뷰 리스트를 추출함
                all_comments.extend(comment_list)  # 모든 리뷰 리스트에 추가함
                has_next = comment_datas.get('hasNext', False)  # 다음 페이지가 있는지 여부를 확인함
                if has_next:
                    comment_id = comment_list[-1]['commentid']  # 다음 페이지가 있으면 마지막 리뷰의 ID를 설정함

            all_reviews.extend(all_comments)  # 모든 리뷰 리스트에 추가함

    return all_reviews  # 모든 리뷰를 반환함


In [7]:
stations = ["강남역", "홍대입구역"]

reviews_data = collect_reviews(stations)

[{'address_name': '서울 강남구 역삼동 817-31', 'category_group_code': 'FD6', 'category_group_name': '음식점', 'category_name': '음식점 > 아시아음식 > 동남아음식 > 베트남음식', 'distance': '', 'id': '1238400864', 'phone': '02-554-8892', 'place_name': '땀땀', 'place_url': 'http://place.map.kakao.com/1238400864', 'road_address_name': '서울 강남구 강남대로98길 12-5', 'x': '127.0279785223454', 'y': '37.50040282892629'}, {'address_name': '서울 강남구 역삼동 822-4', 'category_group_code': 'FD6', 'category_group_name': '음식점', 'category_name': '음식점 > 일식 > 초밥,롤', 'distance': '', 'id': '13575898', 'phone': '02-2051-1477', 'place_name': '갓덴스시 강남점', 'place_url': 'http://place.map.kakao.com/13575898', 'road_address_name': '서울 강남구 테헤란로 109', 'x': '127.029090699483', 'y': '37.498777145173'}, {'address_name': '서울 서초구 서초동 1317-31', 'category_group_code': 'FD6', 'category_group_name': '음식점', 'category_name': '음식점 > 중식 > 중국요리', 'distance': '', 'id': '8279464', 'phone': '02-534-2783', 'place_name': '딘타이펑 강남점', 'place_url': 'http://place.map.kakao.com/827

In [8]:
import pandas as pd
reviews_df = pd.DataFrame(reviews_data)  # 리뷰 데이터를 DataFrame으로 변환함
reviews_df.to_csv('/Users/khb43/Desktop/HANKYUNG_WITH_TOSS_BANK(2)/week8/week8(3) 실무프로젝트 연습/reviews.csv', index=False)  # CSV 파일로 저장함

In [9]:
reviews_df.shape

(6983, 23)

# 전처리
1. Step 1: 한글 데이터만 남기기
2. Step 2: 추가 전처리 작업
3. Step 3: 전처리된 데이터를 CSV로 저장

In [10]:
import pandas as pd
import re

# 리뷰 데이터 불러오기
reviews_df = pd.read_csv('/Users/khb43/Desktop/HANKYUNG_WITH_TOSS_BANK(2)/week8/week8(3) 실무프로젝트 연습/reviews.csv')  # 수집된 리뷰 데이터를 불러옴

# 한글 데이터만 남기기
def preprocess_text(text):
    return re.sub("[^ㄱ-ㅎㅏ-ㅣ가-힣 ]", "", str(text))  # 한글과 공백만 남기고 모두 제거함

reviews_df['preprocessed_comment'] = reviews_df['contents'].apply(preprocess_text)  # 리뷰 데이터를 전처리함
reviews_df = reviews_df.dropna(subset=['preprocessed_comment'])  # 전처리된 리뷰 데이터에서 결측값을 제거함
reviews_df = reviews_df[reviews_df['preprocessed_comment'].str.strip() != '']  # 공백만 있는 리뷰를 제거함

# 추가 전처리 작업 (예: 불용어 제거, 길이 필터링 등)
# 불용어 리스트 (예시)
stopwords = ["은", "는", "이", "가", "을", "를", "에", "의", "도", "로", "하다", "있다", "되다"]

# 불용어 제거 함수
def remove_stopwords(text):
    words = text.split()
    return ' '.join([word for word in words if word not in stopwords])

# 불용어 제거 적용
reviews_df['preprocessed_comment'] = reviews_df['preprocessed_comment'].apply(remove_stopwords)

# 전처리된 데이터를 CSV로 저장
reviews_df.to_csv('/Users/khb43/Desktop/HANKYUNG_WITH_TOSS_BANK(2)/week8/week8(3) 실무프로젝트 연습/preprocessed_reviews.csv', index=False)  # 전처리된 리뷰 데이터를 CSV 파일로 저장함

print("전처리 완료 및 파일 저장 완료!")

전처리 완료 및 파일 저장 완료!


# 모델링
1. Step 1: TF-IDF Vectorizer 적용
2. Step 2: 앙상블 모델 사용
3. Step 3: GridSearch를 통한 최적의 하이퍼 파라미터 탐색 및 저장

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

# 전처리된 데이터 불러오기
reviews_df = pd.read_csv('/Users/khb43/Desktop/HANKYUNG_WITH_TOSS_BANK(2)/week8/week8(3) 실무프로젝트 연습/preprocessed_reviews.csv')  # 전처리된 리뷰 데이터를 불러옴

# 리뷰 평점을 감정(target)으로 설정
# 예시로 4점 이상은 긍정(0), 3점은 중립(1), 3점 미만은 부정(2)으로 분류
def get_sentiment(point):
    if point >= 4:
        return 0  # 긍정
    elif point >= 3:
        return 1  # 중립
    else:
        return 2  # 부정

reviews_df['sentiment'] = reviews_df['point'].apply(get_sentiment)

# TF-IDF Vectorizer 적용
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split())  # 띄어쓰기를 기준으로 토큰화함
X = tfidf_vectorizer.fit_transform(reviews_df['preprocessed_comment'])  # 리뷰 데이터를 TF-IDF 벡터로 변환함
y = reviews_df['sentiment']  # 리뷰의 감정을 타겟 값으로 설정함

# 학습 및 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 학습 세트와 테스트 세트로 분리함

# 앙상블 모델 사용
rf = RandomForestClassifier()  # 랜덤 포레스트 모델을 생성함
gb = GradientBoostingClassifier()  # 그레디언트 부스팅 모델을 생성함
ensemble_model = VotingClassifier(estimators=[('rf', rf), ('gb', gb)], voting='soft')  # 두 모델을 앙상블 모델로 결합함

# GridSearch를 통한 최적의 하이퍼 파라미터 탐색
param_grid = {
    'rf__n_estimators': [50, 100],  # 랜덤 포레스트의 트리 수를 설정함
    'gb__n_estimators': [50, 100]  # 그레디언트 부스팅의 트리 수를 설정함
}

grid_search = GridSearchCV(ensemble_model, param_grid, cv=3, n_jobs=-1, verbose=2)  # 그리드 서치를 설정함
grid_search.fit(X_train, y_train)  # 그리드 서치를 수행함

# 최적의 하이퍼 파라미터 저장
best_params = grid_search.best_params_  # 최적의 하이퍼 파라미터를 가져옴
pd.DataFrame([best_params]).to_csv('/Users/khb43/Desktop/HANKYUNG_WITH_TOSS_BANK(2)/week8/week8(3) 실무프로젝트 연습/best_params.csv', index=False)  # 최적의 하이퍼 파라미터를 CSV 파일로 저장함

print(f"Best Parameters: {best_params}")  # 최적의 하이퍼 파라미터를 출력함

# 최적의 하이퍼 파라미터를 적용하여 모델 재 학습
rf.set_params(n_estimators=best_params['rf__n_estimators'])  # 랜덤 포레스트 모델에 최적의 파라미터를 설정함
gb.set_params(n_estimators=best_params['gb__n_estimators'])  # 그레디언트 부스팅 모델에 최적의 파라미터를 설정함
ensemble_model = VotingClassifier(estimators=[('rf', rf), ('gb', gb)], voting='soft')  # 최적의 파라미터로 앙상블 모델을 재생성함
ensemble_model.fit(X_train, y_train)  # 모델을 재학습함

# 테스트 세트로 일반화 오차 확인
y_pred = ensemble_model.predict(X_test)  # 테스트 세트에서 예측을 수행함
test_score = accuracy_score(y_test, y_pred)  # 테스트 세트에서 모델의 성능을 평가함
print(f"Test Score: {test_score}")  # 테스트 세트에서의 성능을 출력함


/opt/anaconda3/envs/ml-env/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Fitting 3 folds for each of 4 candidates, totalling 12 fits


/opt/anaconda3/envs/ml-env/lib/python3.8/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/opt/anaconda3/envs/ml-env/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/anaconda3/envs/ml-env/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/anaconda3/envs/ml-env/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/o

[CV] END ...........gb__n_estimators=50, rf__n_estimators=50; total time=  44.6s
[CV] END ...........gb__n_estimators=50, rf__n_estimators=50; total time=  45.6s
[CV] END ...........gb__n_estimators=50, rf__n_estimators=50; total time=  46.1s
[CV] END ..........gb__n_estimators=50, rf__n_estimators=100; total time=  54.3s
[CV] END ..........gb__n_estimators=50, rf__n_estimators=100; total time=  55.1s
[CV] END ..........gb__n_estimators=50, rf__n_estimators=100; total time=  55.6s
[CV] END ..........gb__n_estimators=100, rf__n_estimators=50; total time= 1.1min
[CV] END ..........gb__n_estimators=100, rf__n_estimators=50; total time= 1.1min
[CV] END ..........gb__n_estimators=100, rf__n_estimators=50; total time= 1.1min
[CV] END .........gb__n_estimators=100, rf__n_estimators=100; total time= 1.2min
[CV] END .........gb__n_estimators=100, rf__n_estimators=100; total time=  48.4s
[CV] END .........gb__n_estimators=100, rf__n_estimators=100; total time=  47.7s
Best Parameters: {'gb__n_est

# 재학습

1. Step 1: 최적의 하이퍼 파라미터를 사용한 모델 훈련
2. Step 2: 테스트 세트로 일반화 오차 확인


In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import train_test_split

# 최적의 하이퍼 파라미터 설정
best_params = {'gb__n_estimators': 100, 'rf__n_estimators': 50}

# 전처리된 데이터 불러오기
reviews_df = pd.read_csv('/Users/khb43/Desktop/HANKYUNG_WITH_TOSS_BANK(2)/week8/week8(3) 실무프로젝트 연습/preprocessed_reviews.csv')  # 전처리된 리뷰 데이터를 불러옴

# TF-IDF Vectorizer 적용
tfidf_vectorizer = TfidfVectorizer(tokenizer=lambda x: x.split())  # 띄어쓰기를 기준으로 토큰화함
X = tfidf_vectorizer.fit_transform(reviews_df['preprocessed_comment'])  # 리뷰 데이터를 TF-IDF 벡터로 변환함
y = reviews_df['point']  # 리뷰의 평점을 타겟 값으로 설정함

# 학습 및 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 학습 세트와 테스트 세트로 분리함

# 최적의 하이퍼 파라미터를 적용하여 모델 재 학습
rf = RandomForestClassifier(n_estimators=best_params['rf__n_estimators'])  # 랜덤 포레스트 모델에 최적의 파라미터를 설정함
gb = GradientBoostingClassifier(n_estimators=best_params['gb__n_estimators'])  # 그레디언트 부스팅 모델에 최적의 파라미터를 설정함
ensemble_model = VotingClassifier(estimators=[('rf', rf), ('gb', gb)], voting='soft')  # 최적의 파라미터로 앙상블 모델을 재생성함

# 모델 훈련
ensemble_model.fit(X_train, y_train)  # 모델을 재학습함

# 테스트 세트로 일반화 오차 확인
test_score = ensemble_model.score(X_test, y_test)  # 테스트 세트에서 모델의 성능을 평가함
print(f"Test Score: {test_score}")  # 테스트 세트에서의 성능을 출력함


/opt/anaconda3/envs/ml-env/lib/python3.8/site-packages/sklearn/feature_extraction/text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Test Score: 0.6062925170068028


예측 좀 치냐?

In [46]:
from konlpy.tag import Okt

# 감정 예측 함수 정의
okt = Okt()  # 형태소 분석기 초기화

def sentiment_predict(sentence):
    # 문장을 형태소 분석하여 어간 추출 및 정규화
    sentence_norm_stem = okt.morphs(sentence, stem=True, norm=True)
    # 추출된 형태소를 공백으로 연결하여 하나의 문자열로 변환
    sentence_test = ' '.join(sentence_norm_stem)
    # 변환된 문장을 TF-IDF 벡터로 변환
    text_vector = tfidf_vectorizer.transform([sentence_test])
    # 학습된 모델을 사용하여 감정 예측
    pred = ensemble_model.predict(text_vector)
    
    # 예측 결과 출력
    print(sentence, "====>", pred)



In [49]:
# 감정 예측 예시
sentiment_predict("사장님 불친절")

사장님 불친절 ====> [1]


In [53]:
sentiment_predict('보통입니다')

보통입니다 ====> [3]


In [48]:
# 감정 예측 예시
sentiment_predict("위생은 좀 더럽지만 맛은 좋아요")


위생은 좀 더럽지만 맛은 괜찮았어요 ====> [4]


In [50]:
sentiment_predict("와 개맛있다.......")

와 개맛있다....... ====> [5]


In [51]:
sentiment_predict("다시는 먹고 싶지 않은 맛")

다시는 먹고 싶지 않은 맛 ====> [5]


In [52]:
sentiment_predict("음식 정말 맛 없어요")

음식 정말 맛 없어요 ====> [5]
